In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer, make_column_selector
from sklearn.metrics import r2_score, log_loss, accuracy_score, roc_auc_score

from sklearn.svm import SVC
from tqdm import tqdm  # Provides the progress of model running
import os
import warnings
warnings.filterwarnings("ignore")

In [34]:
img = pd.read_csv('D:/Machine_Learning/Cases/Image_Segmentation/Image_Segmentation.csv')
img


,Class,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,rawred.mean,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean
0,BRICKFACE,188,133,9,0.000000,0.0,0.333333,0.266667,0.500000,0.077778,6.666666,8.333334,7.777778,3.888889,5.000000,3.333333,-8.333333,8.444445,0.538580,-0.924817
1,BRICKFACE,105,139,9,0.000000,0.0,0.277778,0.107407,0.833333,0.522222,6.111111,7.555555,7.222222,3.555556,4.333334,3.333333,-7.666666,7.555555,0.532628,-0.965946
2,BRICKFACE,34,137,9,0.000000,0.0,0.500000,0.166667,1.111111,0.474074,5.851852,7.777778,6.444445,3.333333,5.777778,1.777778,-7.555555,7.777778,0.573633,-0.744272
3,BRICKFACE,39,111,9,0.000000,0.0,0.722222,0.374074,0.888889,0.429629,6.037037,7.000000,7.666666,3.444444,2.888889,4.888889,-7.777778,7.888889,0.562919,-1.175773
4,BRICKFACE,16,128,9,0.000000,0.0,0.500000,0.077778,0.666667,0.311111,5.555555,6.888889,6.666666,3.111111,4.000000,3.333333,-7.333334,7.111111,0.561508,-0.985811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,GRASS,36,243,9,0.111111,0.0,1.888889,1.851851,2.000000,0.711110,13.333333,9.888889,12.111111,18.000000,-10.333333,-3.666667,14.000000,18.000000,0.452229,2.368311
205,GRASS,186,218,9,0.000000,0.0,1.166667,0.744444,1.166667,0.655555,13.703704,10.666667,12.666667,17.777779,-9.111111,-3.111111,12.222222,17.777779,0.401347,2.382684
206,GRASS,197,236,9,0.000000,0.0,2.444444,6.829628,3.333333,7.599998,16.074074,13.111111,16.666668,18.444445,-8.888889,1.777778,7.111111,18.555555,0.292729,2.789800
207,GRASS,208,240,9,0.111111,0.0,1.055556,0.862963,2.444444,5.007407,14.148149,10.888889,13.000000,18.555555,-9.777778,-3.444444,13.222222,18.555555,0.421621,2.392487


In [35]:
# Encode the target variable
le = LabelEncoder()
img['Class'] = le.fit_transform(img['Class'])

X = img.drop("Class", axis=1)
y = img['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26, stratify=img['Class'])


In [36]:
# 3. Column Transformation
# Handle potential categorical features (though Image_Segmentation is mostly numeric)
ohe = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")

trans = ColumnTransformer(
    transformers=[
        ("OHE", ohe, make_column_selector(dtype_include=object))
    ], 
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

In [37]:
scalar = StandardScaler()
X_trn_scl = scalar.fit_transform(X_trn_ohe)
X_tst_scl = scalar.transform(X_tst_ohe)

In [39]:
Cs=np.linspace(0.001,5,20)
dfs = ['ovo','ovr']
scores=[]
for c in tqdm(Cs):
    for f in dfs :
        svm=SVC(kernel='linear', probability=True, random_state=26, C=c, decision_function_shape=f)
        svm.fit(X_trn_scl, y_train)
    
        y_pred_prob = svm.predict_proba(X_tst_scl)
    
        scores.append([c,f, log_loss(y_test, y_pred_prob)])
    
df_scores = pd.DataFrame(scores, columns=['C','dfs', 'score' ])
df_scores.sort_values(['score'], ascending=True).head()

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 46.87it/s]


,C,dfs,score
3,0.264105,ovr,0.629300
2,0.264105,ovo,0.629300
5,0.527211,ovr,0.661606
4,0.527211,ovo,0.661606
6,0.790316,ovo,0.679109


In [41]:
Cs=np.linspace(0.001,5,20)
dfs = ['ovo','ovr']
Gs=np.linspace(0.001,5,20)
scores=[]
for c in tqdm(Cs):
    for f in dfs :
        for g in Gs :
            svm=SVC(kernel='rbf', probability=True, random_state=26, C=c, gamma=g, decision_function_shape=f)
            svm.fit(X_trn_scl, y_train)
        
            y_pred_prob = svm.predict_proba(X_tst_scl)
        
            scores.append([c,f,g, log_loss(y_test, y_pred_prob)])
    
df_scores = pd.DataFrame(scores, columns=['C','dfs','gamma', 'score' ])
df_scores.sort_values(['score'], ascending=True).head()

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:15<00:00,  1.27it/s]


,C,dfs,gamma,score
241,1.579632,ovo,0.264105,0.491097
261,1.579632,ovr,0.264105,0.491097
301,1.842737,ovr,0.264105,0.492866
281,1.842737,ovo,0.264105,0.492866
221,1.316526,ovr,0.264105,0.493293
